# Entrenament model offline - Predicció glucosa

El codi, el preprocessament i l'entrenament s'han fet tenint en compte la exploració de dades del dataset realitzant previament, per tal d'entrenar els models de forma clara i estructurada.

#### Import de les llibreries necessaries

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error


import warnings

warnings.simplefilter("ignore", FutureWarning)

pd.set_option('display.max_columns', None)

#### Definició de les columnes dels datasets

In [33]:
cols = [
    'year', 'month', 'day', 'hour', 'minute', 'second', # A–F
    'glucose_level', # G
    'finger_stick', # H
    'basal', # I
    'bolus', # J
    'sleep', # K
    'work', # L
    'stressors', # M
    'hypo_event', # N
    'illness', # O
    'exercise', # P
    'basis_heart_rate', # Q
    'basis_gsr', # R
    'basis_skin_temperature', # S
    'basis_air_temperature',  # T
    'basis_step', # U
    'basis_sleep', # V
    'meal', # W
    'meal_type' # X
]

#### Definició de pacients i horitzons

In [34]:
# Definicio de numero dels pacients
PACIENTS = [559, 563, 570, 575, 588, 591]

# Definició de l'horitzo
HORITZO = {30:6, 60:12}  # minuts : files (pas de 5 mins)

## Carrega del dataset

In [35]:
# Funcio per obtenir els datasets dels pacients
def load_data(pacient, train_or_test):
    df=pd.read_csv(f'../data/{pacient}/{pacient}_{train_or_test}.csv', sep=';', header = None, names = cols)
    return df

## Preprocessament del dataset

In [36]:
def preprocess(df):
    prep = df.copy()

    # Afegim un time per unificar la dada temporal
    prep['time'] = pd.to_datetime(df[['year', 'month', 'day', 'hour', 'minute']])
    prep = prep.drop(columns=['second']) # ja que aquesta columna no aporta valor

    # Ens asegurem que estiguin ordenades de forma cronologica
    prep.sort_values('time', inplace=True)

    # Coma decimal a punt
    # He observat que aqueste categories estan mal asignades com a object, farem el canvi de coma a punt perque detacti el decimal
    # Convertirem les columnes a numeriques
    convert = ["basal","bolus","basis_gsr","basis_skin_temperature","basis_air_temperature"]
    
    for c in convert:
        prep[c] = (prep[c].astype(str)
                   .str.replace(",",".", regex=False)
                   .str.strip()
                   .astype(float))

    # Unifiquem el tipo de meal que han fet
    cat_meal = {
        1:"Desayuno",
        2:"Almuerzo",
        3:"Cena",
        4:"Snack",
        5:"Correccion_hipo"
    }

    prep["meal_type"] = prep["meal_type"].map(cat_meal).astype("category")

    # Obtenim les categories en columnes separades. 
    prep = pd.get_dummies(prep, columns=['meal_type'], dummy_na=False, prefix='meal')


    
    # Tot hi que basal també sigui un error que sigui 0, no la imputarem degut a possibles errors, per tant tampoc ho asignarem com un valor no valid
    # Zeros que no poden ser valids
    invalid_zero = [
        "glucose_level",
        "basis_heart_rate",
        "basis_gsr",
        "basis_skin_temperature",
        "basis_air_temperature"
    ]
    
    # Reemplaçem les caracteristiques que no poden ser 0 per valors null (per posteriorment imputarles)
    prep[invalid_zero] = prep[invalid_zero].replace(0, np.nan)

    imputacio = [
        "basis_heart_rate",
        "basis_gsr",
        "basis_skin_temperature",
        "basis_air_temperature"
    ]
    # Fem servir forward-fill per tal d'asegurarnos que no es mira al futur
    # ffill sense limits ja que ens dona millors resultats en la predicció
    prep[imputacio] = prep[imputacio].fillna(method='ffill')

    prep = prep.dropna(subset=['glucose_level'])
    
    # Tornem a treure la columna time ja que no son valors entrenables
    prep = prep.drop(columns=['time'])
    return prep

## Entrenament del model

#### Definició de la funcio de split features i target

In [37]:
def make_xy(df, files):

    # Definim y com al nivell de glucosa a predir
    # Agafarem el valor a pedir "x" files més amunt segons l'horitzo (30 mins: 6 files o 60 min: 12 files)
    y = df['glucose_level'].shift(-files)

    # Definim X amb totes les columnes pero sense les ultimes files, ja que no tindran predicció y
    X = df.iloc[:-files].copy()

    # Ajustem la y perque tingui el mateix nombre de files que x (eliminant les ultimes files ja que no poden ser predites)
    y = y.iloc[:-files]

    return X, y

#### Definició de la funció de evaluació del model

In [38]:
def evaluate(y_true, y_pred):
    # Calculem el rmse i mae donat el y_true i el y_pred (valor que ha predit el model vs el real)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return rmse, mae

### Entrenament per cada pacient (model offline)

In [39]:
resultats = []

for pacient in PACIENTS:
    print(f'\nPacient {pacient}')

    # Cridem la funció load_data per carregar les dades sense processar.
    train_raw = load_data(pacient, 'train')
    test_raw  = load_data(pacient, 'test')

    # Preprocessem les dades tal i com hem definit anteriorment, per tal que no hi hagi dades buides i fer una bona imputació
    # per tal que el model "no miri al futur"
    train = preprocess(train_raw)
    test  = preprocess(test_raw)

    # Igualem les columnes del train y del test i emplenem el que falti amb 0.
    # Inicialment el train i el test tenen les mateixes features (mateixes columnes 24), al fer el preprocessament pot ser que al codificar
    # la columna de meal_type no estiguin totes les categories apuntades en el test. Per tant en cas que n'hi hagi una la quan no 
    # s'hagi assignat en el test la imputarem amb valor 0, utilitzant outer join.
    # He observat que el test no te la categoria de corrección hipo i per tant te una columna menys, alhora de preprocessar.
    # La mantindrem en el train ja que igualment ajuda el model a fer una millor deteccio de la glucosa.
    train, test = train.align(test, join='outer', axis=1, fill_value=0)

    # Ignorem els 60 minuts primers del test ja que ni podrien ser predits i es podria correlacionar amb les ultimes files del train
    test = test.iloc[12:].reset_index(drop=True)

    for minuts, files in HORITZO.items():

        # Entrenament del model
        X_train, y_train = make_xy(train, files)
        
        # No s'utilitza un gridSearch ja que suposaria més temps d'entrenament i capacitat de computació
        # Amb iteració d'alguns dels paramentres del RF, els actuals m'aporten valors bons.
        model = RandomForestRegressor(
            n_estimators=1000,
            max_depth=10,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)


        # Predicció del test
        # Treiem les ultimes files segons l'horitzo, ja que no podran ser predites
        X_test = test.iloc[:-files].copy()

        # El true del test, serà el desplaçament cap amunt de les files.
        y_true = test['glucose_level'].shift(-files)

        # Finalment també treiem les files que no han pogut ser predites menors a l'horitzo
        y_true = y_true.iloc[:-files].reset_index(drop=True)

        # Fem la predicció del model que hem entrenat
        y_pred = model.predict(X_test)



        # guardem la predicció feta a csv
        directori_pred = f'../data/predicted/pred{pacient}_{minuts}min.csv'

        # Fem index més 12 per tal de poder comparar les prediccions visualment amb més facilitat (anteriorment hem tret 1 hora del test)
        df_pred = pd.DataFrame({'Index': X_test.index + 6, f'pred_glucosa_t+{minuts}': y_pred}) # Comença als 30 mins
        df_pred.to_csv(directori_pred, index=False)

            

        # Evaluació del model
        # Cridem la funció que hem definit anteriorment de evaluació que ens retorna rmse i mae
        rmse, mae = evaluate(y_true, y_pred)

        # Afegim els restats en un diccionari per posteriorment poder-los mostrar
        resultats.append({
            'Pacient': pacient,
            'Horitzo': minuts,
            'RMSE': rmse,
            'MAE' : mae
        })


        print(f'{minuts} min: RMSE={rmse:.2f}  MAE={mae:.2f}')




Pacient 559
30 min: RMSE=25.17  MAE=17.94
60 min: RMSE=38.46  MAE=28.61

Pacient 563
30 min: RMSE=21.14  MAE=15.62
60 min: RMSE=34.24  MAE=25.44

Pacient 570
30 min: RMSE=19.41  MAE=13.81
60 min: RMSE=32.49  MAE=24.45

Pacient 575
30 min: RMSE=25.00  MAE=18.76
60 min: RMSE=43.98  MAE=34.61

Pacient 588
30 min: RMSE=22.05  MAE=16.21
60 min: RMSE=33.63  MAE=24.90

Pacient 591
30 min: RMSE=25.24  MAE=19.23
60 min: RMSE=39.34  MAE=31.55


## Taula final de resutats

In [40]:
resultats_df = pd.DataFrame(resultats)

taula = (resultats_df.pivot(index='Pacient', columns='Horitzo', values=['RMSE','MAE']))
print(f'Visualització inicial de la taula: \n{taula}')

# Renombrem les columnes de la taula
taula.columns = ['RMSE 30','RMSE 60','MAE 30', 'MAE 60']

# Reordenem les columnes
taula = taula[['RMSE 30','MAE 30','RMSE 60','MAE 60']]

# Calculem el promig
promig = taula.mean().to_frame().T
promig.index = ['PROMIG'] 

# Mostrem la taula final amb el promig
taula_final = pd.concat([taula, promig], axis=0)

print("\nRESULTATS FINALS:")
print(taula_final)

Visualització inicial de la taula: 
              RMSE                   MAE           
Horitzo         30         60         30         60
Pacient                                            
559      25.166162  38.459883  17.940706  28.605622
563      21.144070  34.243495  15.619662  25.435914
570      19.406101  32.492189  13.807863  24.450644
575      24.995790  43.980118  18.755061  34.611827
588      22.050246  33.627830  16.214166  24.902954
591      25.243385  39.342509  19.230948  31.546191

RESULTATS FINALS:
          RMSE 30     MAE 30    RMSE 60     MAE 60
559     25.166162  17.940706  38.459883  28.605622
563     21.144070  15.619662  34.243495  25.435914
570     19.406101  13.807863  32.492189  24.450644
575     24.995790  18.755061  43.980118  34.611827
588     22.050246  16.214166  33.627830  24.902954
591     25.243385  19.230948  39.342509  31.546191
PROMIG  23.000959  16.928068  37.024337  28.258859
